# OmniBird — BigBird per-event encoder + cross-attention centroid pool (NeRF positions)

A clean restart on top of the baseline `OmniBird_train_cifar10_dvs.ipynb`.

**Architecture (symmetric, EMA from context → target):**

| Stage | Operation | Shape |
|---|---|---|
| 1. Tokenize | per-event MLP on signal + γ_NeRF(coord) | (B, N, D) |
| 2. Encode | **BigBird** block-sparse self-attention, with a *different* space-filling curve picked per layer (z, z-rev, Hilbert, Hilbert-rev) — receptive field is globally connected through depth | (B, N, D) |
| 3. Pool  | **Cross-attention**: data-conditional **centroids** as queries (positions encoded the same way, γ_NeRF), per-event features as keys/values → one latent per centroid | (B, P, D) |

**Positional embedding policy** (post-debug):
- **Encoder + pool**: one shared `FixedPosEmbedder` (NeRF 5 octaves → frozen orthogonal → d_model). Same fixed mapping for the Tokenizer, the per-layer attention-input bias, and the CentroidPool query.
- **Predictor**: its OWN trainable γ + Linear (untied). Without this, the predictor + target form a closed-form position-only shortcut and the encoder collapses.
- **`reinject_pos=True` is now pre-norm**: pos is added to the attention *input* under LN, not into the residual stream. Earlier code mutated the residual and accumulated `L·pos_emb` through depth — that was direction-collapsing the encoder.

**Crucial pieces we keep:**
* **Space-filling curve serialization** — gives BigBird windowing spatial locality and rotating-curve global coverage.
* **i-JEPA multi-block masking at the patch level** — 4 target blocks × 16 contiguous patches; remaining real patches form the context.

**Robustness lives in the architecture, not the loss.** The CentroidPool now uses **`LocalCrossAttention`** with a spatial-locality bias on attention scores: `scores[q,k] -= α · ‖centroid(q) − coord(k)‖²`. This forces each centroid query to attend to events *near it* in the (x, y, t) cloud, regardless of what Q·K alignment learns. Constant-across-centroids outputs become structurally impossible — different centroids have different local neighborhoods, so the pool output cannot be the same across centroids unless the events themselves are uniform (which they aren't).

With locality guaranteed, both **context AND target** can use the same cross-attention pool (symmetric). The predictor's job becomes the actual i-JEPA contract: map context-side pool features at context centroids to target-side pool features at target centroids — same statistic, different locations. **No regularizer** (`vicreg_var_coef=0`).


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig,
    BigBirdEventEncoderWithPool, PerceiverPredictor, FixedPosEmbedder,
    OmniBirdPatchDataset,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, diag_dict, fmt_diag,
    save_atomic, ensure_dir, short_params, count_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

cfg = OmniBirdConfig()
cfg.coord_dim       = 3
cfg.signal_dim      = 2                # one-hot ON/OFF
cfg.side            = 64
cfg.n_events_total  = 0
cfg.n_events_max    = 8192             # P_max = 256

# Positions: NeRF γ → fixed orthogonal projection → d_model, shared across
# encoder Tokenizer, encoder per-layer pos residual, CentroidPool centroid
# query, and predictor mask-token positioning. Single frozen mapping → no
# tailored position-only shortcut possible.
cfg.pos_embed       = "nerf"
cfg.nerf_freqs      = 5
# Predictor must operate at d_model so the shared embedder works directly.
cfg.d_pred          = 256          # = cfg.d_model

# Patch / centroid layout
cfg.patch_mode         = True
cfg.patch_size         = 32            # K events per patch
cfg.patches_per_block  = 16
cfg.n_pred_blocks      = 4
cfg.n_tgt              = cfg.n_pred_blocks * cfg.patches_per_block      # 64
cfg.ctx_max_patches    = 64
cfg.patch_curve        = "hilbert"

# BigBird sparse attention
cfg.attention_type   = "bigbird"
cfg.block_size       = 8
cfg.window           = 1
cfg.n_random         = 2
cfg.n_global         = 2
# Re-add the positional embedding at every encoder layer. Required here:
# 6 BigBird layers of global mixing wash out the input position signal, so
# event tokens at layer 6 are nearly position-agnostic. Then the cross-
# attention pool's centroid queries can't select local events (attention is
# uniform → output ≈ mean event feature) and every centroid in the same
# sample reads back the same averaged content → collapse.
cfg.reinject_pos     = True

# JEPA
# smooth_l1 (not cosine): cosine only constrains direction, which collapses
# easily to a constant. smooth_l1 forces the predictor to match magnitudes
# too — harder to satisfy with a position-only shortcut.
cfg.loss_type        = "smooth_l1"
# DINO-style target centering: EMA of the per-feature batch mean, subtracted
# before per-token LayerNorm. The working PBB v2 notebook uses this; the
# pointjepa notebook was missing it. LayerNorm alone allows the "all targets
# point in the same direction across batch" minimum (unit variance per dim,
# but colinear across batch). Centering kills that minimum.
cfg.use_centering    = True
cfg.ema_start        = 0.999
cfg.ema_end          = 0.9999
# VICReg off — robustness now comes from the architecture (LocalCrossAttention
# spatial-locality bias), not from a regularizer.
cfg.vicreg_var_coef  = 0.0
# Spatial-locality bias coefficient in the CentroidPool's LocalCrossAttention.
# `scores -= α · ‖q_coord − k_coord‖²` so attention is structurally peaked
# at events near each centroid. Constant-across-centroids outputs become
# impossible because different centroids have different local neighborhoods.
cfg.local_bias_scale = 10.0

# Optim
cfg.epochs           = 100
cfg.batch_size       = 64
cfg.lr               = 8e-4
cfg.probe_interval   = 10
cfg.probe_epochs     = 3
cfg.log_every        = 25
cfg.ckpt_dir         = "./checkpoints_omnibird_cifar10_dvs_xattn"
ensure_dir(cfg.ckpt_dir)

print(f"P_max = {cfg.n_events_max // cfg.patch_size}  K = {cfg.patch_size}")
print(f"context patches = {cfg.ctx_max_patches}  target patches = {cfg.n_tgt}")


## 2. CIFAR10-DVS dataset

In [ ]:
class CIFAR10DVSFromClips:
    def __init__(self, root, sensor_hw=(128, 128)):
        self.root = root; self.h, self.w = sensor_hw
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0:
            ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0: p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f: label = int(f.read().strip())
        return out, label


base = CIFAR10DVSFromClips(DATA_ROOT)
n = len(base); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]
class _Subset:
    def __init__(self, b, i): self.b=b; self.i=i
    def __len__(self): return len(self.i)
    def __getitem__(self, k): return self.b[int(self.i[k])]
base_train = _Subset(base, train_idx); base_test = _Subset(base, test_idx)

from torch.utils.data import DataLoader
mk = lambda b, tr: OmniBirdPatchDataset(b, cfg, train=tr,
                                          patches_per_block=cfg.patches_per_block,
                                          n_pred_blocks=cfg.n_pred_blocks,
                                          ctx_max=cfg.ctx_max_patches,
                                          patch_size=cfg.patch_size,
                                          curve=cfg.patch_curve)
train_ds, train_eval_ds, test_ds = mk(base_train, True), mk(base_train, False), mk(base_test, False)
train_loader      = DataLoader(train_ds,      batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
test_loader       = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, num_workers=0)
print(f"loaded CIFAR10-DVS: train={len(train_ds)}  test={len(test_ds)}")


## 3. Tokenization & masking visualization

Three figures for one clip:
1. Raw events colored by polarity → Hilbert-sorted by curve rank → grouped into patches.
2. Patch centroids overlaid on the cloud (these become the cross-attention queries).
3. i-JEPA-style masking: 4 target blocks (4 colors) and the ~64 context patches.


In [ ]:
SAMPLE_IDX = 0
sample = train_ds[SAMPLE_IDX]
classes = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"sample idx={SAMPLE_IDX}, label={classes[sample['label']]}")

P    = sample["pool_patch_events"].shape[0]
K    = sample["pool_patch_events"].shape[1]
Fdim = sample["pool_patch_events"].shape[2]
pool_ev   = sample["pool_patch_events"].numpy()
centroids = sample["pool_patch_centroids"].numpy()
patch_kpm = sample["pool_patch_kpm"].numpy()
ev_kpm    = sample["pool_event_kpm"].numpy()
ctx_idx   = sample["ctx_patch_idx"].numpy()
tgt_idx   = sample["tgt_patch_idx"].numpy()

events_flat = pool_ev.reshape(P*K, Fdim)
coords = events_flat[:, :3]
pol    = events_flat[:, 3] - events_flat[:, 4]                # +1 ON, -1 OFF, 0 pad
ev_kpm_flat = ev_kpm.reshape(P*K)
real = ~ev_kpm_flat
real_coords = coords[real]; real_pol = pol[real]
print(f"P={P}  K={K}  real events={real.sum()}  real patches={(~patch_kpm).sum()}")


In [ ]:
# Figure 1: raw → Hilbert sort → patches
fig = plt.figure(figsize=(20, 6))

ax = fig.add_subplot(1, 3, 1, projection='3d')
pos, neg = real_coords[real_pol > 0], real_coords[real_pol < 0]
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], s=1, c='red',  alpha=0.4, label='ON')
ax.scatter(neg[:, 0], neg[:, 1], neg[:, 2], s=1, c='blue', alpha=0.4, label='OFF')
ax.set_title(f"(a) raw events ({real.sum()})")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t'); ax.legend(loc='lower right', fontsize=8)

ax = fig.add_subplot(1, 3, 2, projection='3d')
rank = np.arange(P*K)[real]
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=2, c=rank, cmap='turbo', alpha=0.7)
ax.set_title("(b) Hilbert-sorted (color = curve rank)")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')

ax = fig.add_subplot(1, 3, 3, projection='3d')
patch_id = np.repeat(np.arange(P), K)[real]
rng_cmap = np.random.RandomState(0)
patch_colors = plt.cm.tab20(rng_cmap.permutation(P) % 20)
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=3, c=patch_colors[patch_id], alpha=0.8)
ax.set_title(f"(c) patches  (P={P}, K={K})")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
plt.tight_layout(); plt.show()


In [ ]:
# Figure 2: centroids = cross-attention queries
fig = plt.figure(figsize=(14, 6))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=1, c='lightgray', alpha=0.3)
real_cen = centroids[~patch_kpm]
ax.scatter(real_cen[:, 0], real_cen[:, 1], real_cen[:, 2],
           s=25, c='black', marker='o', edgecolors='red', linewidths=0.5)
ax.set_title(f"(a) patch centroids ({(~patch_kpm).sum()})\n"
             f"→ these become CROSS-ATTENTION QUERIES")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')

# Highlight ONE centroid and draw arrows to its top-K events
ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=1, c='lightgray', alpha=0.25)
real_patch_ids = np.where(~patch_kpm)[0]
chosen = real_patch_ids[len(real_patch_ids) // 2]
cen_c = centroids[chosen]
ax.scatter([cen_c[0]], [cen_c[1]], [cen_c[2]],
            s=250, c='orange', marker='*', edgecolors='black', linewidths=1.5,
            label=f'one centroid query (patch {chosen})')
# Visualize that it can attend to ALL events, but its closest 64 are highlighted
d2 = ((real_coords - cen_c) ** 2).sum(-1)
near = np.argsort(d2)[:64]
ax.scatter(real_coords[near, 0], real_coords[near, 1], real_coords[near, 2],
           s=12, c='orange', alpha=0.7, label='64 nearest events (visual)')
ax.set_title("(b) one cross-attn query attends GLOBALLY\nover all encoded events")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t'); ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# Figure 3: i-JEPA multi-block masking (4 target blocks + context)
TGT_COLORS = ['#fbbf24', '#34d399', '#60a5fa', '#f472b6']
fig = plt.figure(figsize=(14, 6))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=1, c='lightgray', alpha=0.25)
for k in range(cfg.n_pred_blocks):
    block_ids = tgt_idx[k*cfg.patches_per_block:(k+1)*cfg.patches_per_block]
    em = np.isin(np.repeat(np.arange(P), K)[real], block_ids)
    ax.scatter(real_coords[em, 0], real_coords[em, 1], real_coords[em, 2],
               s=8, c=TGT_COLORS[k], alpha=0.75,
               label=f'target block {k+1}')
ax.set_title(f"(a) {cfg.n_pred_blocks} target blocks × {cfg.patches_per_block} patches")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.legend(loc='lower right', fontsize=7, ncol=2)

ax = fig.add_subplot(1, 2, 2, projection='3d')
in_ctx = np.isin(np.repeat(np.arange(P), K)[real], ctx_idx)
in_tgt = np.isin(np.repeat(np.arange(P), K)[real], tgt_idx)
leftover = ~(in_ctx | in_tgt)
ax.scatter(real_coords[leftover, 0], real_coords[leftover, 1], real_coords[leftover, 2],
           s=1, c='lightgray', alpha=0.2, label='leftover')
ax.scatter(real_coords[in_ctx, 0], real_coords[in_ctx, 1], real_coords[in_ctx, 2],
           s=2, c='#ef4444', alpha=0.5,
           label=f'context ({(~patch_kpm[ctx_idx]).sum()} real)')
for k in range(cfg.n_pred_blocks):
    block_ids = tgt_idx[k*cfg.patches_per_block:(k+1)*cfg.patches_per_block]
    em = np.isin(np.repeat(np.arange(P), K)[real], block_ids)
    ax.scatter(real_coords[em, 0], real_coords[em, 1], real_coords[em, 2],
               s=4, c=TGT_COLORS[k], alpha=0.6)
ax.set_title("(b) full mask layout\nred=context, colors=targets, gray=leftover")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.legend(loc='lower right', fontsize=7)
plt.tight_layout(); plt.show()


## 4. Models (symmetric, single shared fixed positional embedding)

In [ ]:
# ── Shared fixed positional embedding ────────────────────────────────────
# NeRF γ → orthogonal projection → d_model. Frozen. Used identically in:
#  - context encoder's Tokenizer
#  - context encoder's per-layer pos residual (reinject_pos=True)
#  - context encoder's CentroidPool query
#  - target encoder (deepcopy of the above — same W matrix)
#  - predictor's mask-token positioning + context-pos residual
shared_pos = FixedPosEmbedder(
    coord_dim=cfg.coord_dim, d_model=cfg.d_model,
    num_freqs=cfg.nerf_freqs,
).to(DEVICE)
# Sanity: ensure it has zero trainable params.
n_trainable = sum(p.numel() for p in shared_pos.parameters() if p.requires_grad)
assert n_trainable == 0, f"FixedPosEmbedder has {n_trainable} trainable params"
print(f"FixedPosEmbedder: W is {tuple(shared_pos.W.shape)} (orthogonal, frozen)")

def make_encoder():
    return BigBirdEventEncoderWithPool(
        d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
        n_heads=cfg.n_heads, dim_head=cfg.dim_head,
        block_size=cfg.block_size, window=cfg.window,
        n_random=cfg.n_random, n_global=cfg.n_global,
        ffn_mult=cfg.ffn_mult,
        signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
        fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
        serial_orders=cfg.serial_orders,
        reinject_pos=cfg.reinject_pos,
        side=cfg.side,
        fixed_pos_embedder=shared_pos,
        local_bias_scale=cfg.local_bias_scale,
    )

context_encoder = make_encoder().to(DEVICE)
target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)

# Decisive shortcut test: UNTIE the predictor's positional embedding from the
# encoder's. The predictor gets its own trainable γ_NeRF + Linear projection
# (legacy path). The encoder + pool keep the shared FixedPosEmbedder.
#
# Why: with a single shared fixed projection on both sides, the predictor has
# a closed-form solution — route shared_pos(tgt_cen) straight to proj_out and
# match LN(target_encoder pool output at tgt_cen), which is itself dominated by
# shared_pos(tgt_cen). Giving the predictor its own embedding breaks the trivial
# routing path: its positional mapping is different from the encoder's, so it
# must use cross-attention over g_ctx to recover useful target features.
predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
    pos_embed=cfg.pos_embed, nerf_freqs=cfg.nerf_freqs,
    fixed_pos_embedder=None,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 5. Optim + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

LAST = os.path.join(cfg.ckpt_dir, "omnibird_last.pt")
BEST = os.path.join(cfg.ckpt_dir, "omnibird_best.pt")

RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
m = cfg.ema_start

if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Helpers + probe

In [ ]:
def gather_ctx_subset(batch):
    # Returns (events_flat (B, Pc*K, F), centroids (B, Pc, 3), event_kpm (B, Pc*K))
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    ctx_idx    = batch["ctx_patch_idx"].to(DEVICE)
    B, P, K, Fd = pool_ev.shape
    Pc = ctx_idx.size(1)
    sub_ev    = torch.gather(pool_ev,    1, ctx_idx[..., None, None].expand(B, Pc, K, Fd))
    sub_cen   = torch.gather(pool_cen,   1, ctx_idx[..., None].expand(B, Pc, pool_cen.size(-1)))
    sub_evkpm = torch.gather(pool_evkpm, 1, ctx_idx[..., None].expand(B, Pc, K))
    return sub_ev.reshape(B, Pc*K, Fd), sub_cen, sub_evkpm.reshape(B, Pc*K)


def flatten_pool(batch):
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    B, P, K, Fd = pool_ev.shape
    return pool_ev.view(B, P*K, Fd), pool_cen, pool_evkpm.view(B, P*K)


class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)


def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    probe = LinearProbe(cfg.d_model, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    def _z(b):
        ev, cen, kpm = gather_ctx_subset(b)
        with torch.no_grad():
            enc.eval()
            g = enc(ev, cen, event_kpm=kpm)              # (B, Pc, D)
        return g.mean(dim=1)
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_z(b)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            correct += (probe(_z(b)).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 7. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            tgt_cen = batch["tgt_patch_centroids"].to(DEVICE)

            # Target: full pool of events → encoder + LocalCrossAttention pool
            # with TARGET CENTROIDS as queries. Symmetric with the context side
            # so the predictor maps cross-attn-pool → cross-attn-pool (not pool
            # → mean-pool). The pool's spatial-locality bias makes constant-
            # across-centroids outputs structurally impossible — each centroid
            # query is forced to weight nearby events, and different centroids
            # have different local neighborhoods.
            with torch.no_grad():
                pool_patch_ev   = batch["pool_patch_events"].to(DEVICE)
                pool_evkpm_full = batch["pool_event_kpm"].to(DEVICE)
                B, P, K, Fdim   = pool_patch_ev.shape
                pool_ev_flat    = pool_patch_ev.view(B, P*K, Fdim)
                pool_kpm_flat   = pool_evkpm_full.view(B, P*K)
                h_tgt_raw       = target_encoder(pool_ev_flat, tgt_cen,
                                                  event_kpm=pool_kpm_flat)
                                                                                  # (B, n_tgt, D)
                Dim = h_tgt_raw.size(-1)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (Dim,))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (Dim,))

            # Context: subset of events → encoder; pool with context centroids.
            ctx_ev, ctx_cen, ctx_kpm = gather_ctx_subset(batch)
            g_ctx = context_encoder(ctx_ev, ctx_cen, event_kpm=ctx_kpm)          # (B, Pc, D)

            # Predictor: target mask tokens (at target NeRF positions) cross-attend
            # over context features.
            h_pred = predictor(g_ctx, tgt_cen,
                                ctx_coords=ctx_cen if cfg.predictor_pos_symmetric else None,
                                ctx_key_padding_mask=None)

            jepa_l = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            # VICReg variance hinge — applied to BOTH g_ctx (context encoder
            # output, online, with grad) and h_pred (predictor output, with
            # grad). The previous round computed it on h_tgt — a no_grad
            # tensor — so it had zero gradient and did literally nothing.
            # Now the hinge actually pulls the online encoder away from
            # constant-output collapse.
            g_ctx_flat = g_ctx.reshape(-1, g_ctx.size(-1))
            std_g_ctx  = torch.sqrt(g_ctx_flat.var(dim=0) + 1e-4)
            var_ctx    = F.relu(1.0 - std_g_ctx).mean()
            h_pred_flat = h_pred.reshape(-1, h_pred.size(-1))
            std_h_pred  = torch.sqrt(h_pred_flat.var(dim=0) + 1e-4)
            var_pred    = F.relu(1.0 - std_h_pred).mean()
            var_term = var_ctx + var_pred
            loss = jepa_l + cfg.vicreg_var_coef * var_term

            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                d["jepa_l"]   = jepa_l.item()
                d["var_term"] = var_term.item()
                d["std_ctx"]  = std_g_ctx.mean().item()
                d["std_prd"]  = std_h_pred.mean().item()
                line = fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m)
                line += (f"  jepa={d['jepa_l']:.4f} var={d['var_term']:.3f} "
                         f"std_ctx={d['std_ctx']:.3f} std_prd={d['std_prd']:.3f}")
                print(line)
                history["diag_steps"].append(global_step); history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss,
            "history": history,
        }
        if improved: save_atomic(state, BEST)
        save_atomic(state, LAST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  loss={avg:.4f}  m={m:.4f}  "
              f"{time.time()-t0:.1f}s" + ("  ★" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_p = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_p:.1f}s)")

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 8. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("avg loss"); axes[0].set_title("JEPA loss"); axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    axes[1].plot(steps, cos_mean, color='C2'); axes[1].set_ylim(-0.1, 1.05)
    axes[1].set_xlabel("step"); axes[1].set_title("cos(h_pred, h_tgt)"); axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe acc (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
